# 04 — Sun‑disk centre from the lunar limb + ephemeris

Paper Section 3.2, steps 2–3.  For every frame:

1. camera timestamp (IST) → UTC;
2. Skyfield/DE421 topocentric apparent Sun and Moon → angular radii, separation and
   position angle of the Sun relative to the Moon;
3. plate scale = CHT lunar radius / lunar angular radius; the Sun–Moon vector is
   rotated into the image with the North angle (168°) → Sun centre in pixels.

`config.SUN_CENTER_ROTATION` selects the rotation convention: `"astrometric"`
(validated on the star field in notebook 03, default) or `"legacy"` (the
convention of the original `find_sun_center` notebook, which reproduces the
submitted figures).  Output `products/sun_moon_centers.csv`.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
if config.MOON_CENTERS_SOURCE == "legacy":
    moon = pd.read_csv(config.LEGACY_MOON_CENTERS_CSV); moon["Filename"] = moon.Filename.str.split("/").str[-1]
else:
    moon = pd.read_csv(config.MOON_CENTERS_CSV)
moon = moon.drop_duplicates("Filename").set_index("Filename")      # legacy: first match wins
print(f"Moon centres: {config.MOON_CENTERS_SOURCE} ({len(moon)} rows);  rotation convention: {config.SUN_CENTER_ROTATION}")

In [ ]:
eph = utils.Ephemeris()
rows = []
for f in utils.list_calibrated_frames():
    if f not in moon.index:
        print("no Moon centre for", f); continue
    m = moon.loc[f]
    utc = utils.fits_timestamp_utc(utils.read_header(f))
    moon_r_as, sun_r_as, sep_as, pa = eph.sun_moon_geometry(utc)
    sx, sy, scale = utils.sun_center_from_moon(moon_r_as, sep_as, pa, m.Moon_XC, m.Moon_YC, m.Moon_Radius)
    rows.append(dict(filename=f, moon_xc=m.Moon_XC, moon_yc=m.Moon_YC, moon_radius=m.Moon_Radius,
                     sun_xc=sx, sun_yc=sy, sun_radius=sun_r_as * scale,
                     celestial_north_from_image_y=config.IMAGE_NORTH_ANGLE_FROM_Y_DEG,
                     utc=utc, plate_scale_arcsec_per_px=1.0 / scale, sep_arcsec=sep_as, pa_sun_from_moon_deg=pa,
                     convention=config.SUN_CENTER_ROTATION))
centers = pd.DataFrame(rows)
centers.to_csv(config.SUN_MOON_CENTERS_CSV, index=False, float_format="%.3f")
centers.head()

In [ ]:
# Compare with the table used for the paper (legacy convention).  In astrometric mode the
# difference IS the convention offset: 3-11 px, rotating with the Sun-Moon position angle.
leg = pd.read_csv(config.LEGACY_SUN_MOON_CENTERS_CSV).set_index("filename")
cmp = centers.set_index("filename").join(leg, rsuffix="_legacy", how="inner")
cmp["position"] = [utils.position_from_filename(f) for f in cmp.index]
cmp["dx"] = cmp.sun_xc - cmp.sun_xc_legacy; cmp["dy"] = cmp.sun_yc - cmp.sun_yc_legacy
print(f"Sun centre (this run) - (paper table), px:  max |dx| = {cmp.dx.abs().max():.2f}, max |dy| = {cmp.dy.abs().max():.2f}")
cmp[cmp.position.isin(config.POLARIZER_POSITIONS)].groupby("position")[["dx", "dy", "sep_arcsec", "pa_sun_from_moon_deg"]].mean().round(2)

In [ ]:
# Sun-Moon geometry through totality
pol = centers[[utils.position_from_filename(f) in config.POLARIZER_POSITIONS for f in centers.filename]]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(pd.to_datetime(pol.utc), pol.sep_arcsec, ".", ms=4); ax[0].set_ylabel("Sun-Moon separation [arcsec]")
ax[1].plot(pd.to_datetime(pol.utc), pol.pa_sun_from_moon_deg, ".", ms=4); ax[1].set_ylabel("PA of Sun from Moon [deg]")
for a in ax: a.set_xlabel("UTC"); a.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# QA mosaic: Moon (blue) and Sun (orange) circles + celestial-North arrow, Position 1 frames
import math
show = centers[centers.filename.str.contains("Position1")].sort_values("filename")
ncols = 6; nrows = math.ceil(len(show) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 2.4*nrows))
for ax in axes.flat: ax.axis("off")
n, _ = utils.north_east_vectors()
for ax, (_, r) in zip(axes.flat, show.iterrows()):
    img = fits.getdata(config.CALIBRATED_LIGHTS_DIR / r.filename, memmap=True)
    ax.imshow(utils.asinh_stretch(np.sum(np.asarray(img[:, ::8, ::8], float), 0)), cmap="gray",
              extent=(0, config.IMAGE_SHAPE[1], config.IMAGE_SHAPE[0], 0))
    ax.add_patch(plt.Circle((r.moon_xc, r.moon_yc), r.moon_radius, ec="deepskyblue", fc="none", lw=0.7))
    ax.add_patch(plt.Circle((r.sun_xc, r.sun_yc), r.sun_radius, ec="orangered", fc="none", lw=0.7))
    ax.arrow(1500, 4700, 900 * n[0], 900 * n[1], color="lime", width=40)
    ax.set_title(r.filename.replace("-cal.fits", ""), fontsize=7)
plt.tight_layout()
fig.savefig(config.FIGURES_DIR / "qa_sun_moon_mosaic_position1.png", dpi=120)